# Week 14: LLM Fine-Tuning & Quantization
## Notebook 09: DoRA: Weight-Decomposed Low-Rank Adaptation

### 1. Conceptual Intuition & Theoretical Background
While LoRA decomposes weight updates into low-rank matrices ($W = W_0 + \frac{\alpha}{r} B A$), full fine-tuning modifies both the **magnitude** and **direction** of weight vectors with distinct dynamics. LoRA ties magnitude and direction updates together, limiting expressiveness.

**DoRA (Liu et al., 2024):**
Decomposes weight matrix $W \in \mathbb{R}^{d \times k}$ into a **Magnitude Vector** $m \in \mathbb{R}^{1 \times k}$ and a normalized **Directional Matrix** $V \in \mathbb{R}^{d \times k}$:
$$W = m \odot \frac{V}{\|V\|_c} = m \odot \frac{W_0 + \Delta W}{\|W_0 + \Delta W\|_c}$$

Where:
- $m = \|W_0\|_c$ is a trainable magnitude vector initialized to column norms of $W_0$.
- $\Delta W = B \cdot A$ is the standard low-rank directional update.
- $\|\cdot\|_c$ denotes column-wise vector $L_2$ norm.

Decoupling magnitude and direction enables DoRA to match or exceed full fine-tuning performance while maintaining LoRA's inference efficiency (can be merged back into $W$ with zero added latency).

---

### 2. Scratch Python Implementation: DoRA Layer

In [ ]:
import numpy as np

class DoRALayerScratch:
    def __init__(self, in_features, out_features, rank=4, alpha=8):
        self.r = rank
        self.scaling = alpha / rank
        
        # Frozen base weights W_0
        np.random.seed(42)
        self.W0 = np.random.normal(0, 0.05, (in_features, out_features))
        
        # Trainable Magnitude vector m initialized to column norms of W_0
        self.m = np.linalg.norm(self.W0, axis=0, keepdims=True)
        
        # Trainable Low-Rank Directional Matrices
        self.A = np.random.normal(0, 0.02, (in_features, rank))
        self.B = np.zeros((rank, out_features)) # Zero initialization
        
    def get_effective_weight(self):
        # 1. Compute directional matrix V = W_0 + (alpha / r) * (A @ B)
        delta_W = self.scaling * np.dot(self.A, self.B)
        V = self.W0 + delta_W
        
        # 2. Normalize columns of V
        V_norm = np.linalg.norm(V, axis=0, keepdims=True) + 1e-9
        V_normalized = V / V_norm
        
        # 3. Scale by magnitude vector m
        W_dora = self.m * V_normalized
        return W_dora
        
    def forward(self, x):
        W_eff = self.get_effective_weight()
        return np.dot(x, W_eff)

# Test DoRA Layer
in_dim, out_dim = 16, 8
layer = DoRALayerScratch(in_features=in_dim, out_features=out_dim, rank=2)

x_sample = np.random.randn(2, in_dim)
y_init = layer.forward(x_sample)

# Simulate training update on B
layer.B += np.random.normal(0, 0.01, layer.B.shape)
y_updated = layer.forward(x_sample)

print("--- DoRA Decomposed Low-Rank Adaptation ---")
print(f"Base Weight Shape:        {layer.W0.shape}")
print(f"Magnitude Vector Shape:   {layer.m.shape} (Only {out_dim} parameters!)")
print(f"Directional Rank r:       {layer.r}")
print(f"Initial Output Sample 0:  {np.round(y_init[0][:4], 4)}")
print(f"Updated Output Sample 0:  {np.round(y_updated[0][:4], 4)}")
print("Magnitude and Direction decoupled cleanly for higher representational capacity!")

### 3. Step-by-Step Code Walkthrough
1. **Magnitude Extraction ($m$):** Captures column scale vectors initialized to $\|W_0\|_c$.
2. **Directional Normalization:** Normalizes $W_0 + B A$ to decouple directional shifts from scaling.
3. **Zero Overhead at Deployment:** Can be folded back into a single matrix $W_{\text{final}}$ prior to inference.

---

### 4. Industry Applications & Trade-offs
- **High-Precision LLM Fine-Tuning:** Preferred over LoRA for complex math reasoning and code generation tasks.
- **Trade-off:** Training introduces additional normalization operations that increase backward pass memory slightly.

---

### 5. Top Interview Questions & Answers
**Q1: How does DoRA differ from standard LoRA in weight update dynamics?**
> **Answer:** In LoRA, magnitude and direction updates are proportional and coupled. DoRA normalizes the directional matrix and scales it by a dedicated magnitude vector, allowing magnitude and direction to adapt independently.

**Q2: Can DoRA weights be merged into the base model for zero-latency serving?**
> **Answer:** Yes. After training completes, the effective weight $W_{\text{DoRA}} = m \odot \frac{W_0 + \Delta W}{\|W_0 + \Delta W\|_c}$ can be computed and stored as the new base weight tensor.